In [17]:
import logging
logging.basicConfig(
    filename='log/osf_fetch_log.log',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logging.info("Starting the script")

# OSF API endpoint
BASE_URL = "https://api.osf.io/v2/preprints/"
BASE_PATH = "/home/ec2-user/r2/osf/"
PAGE_SIZE = 200

import requests
import json
from math import ceil
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
load_dotenv()

import os

API_TOKEN = os.getenv('OSF_API_TOKEN')

def write_flush_file(results, path):
    if path is None:
        raise ValueError('Path must be provided to write results to file.')

    jsonl_str = '\n'.join(json.dumps(r, ensure_ascii=False) for r in results)
    with open(path, "a", encoding="utf-8") as f:
        f.write(jsonl_str + "\n")
    return []

def get_total_count(year, provider=None):
    url = BASE_URL
    params = {
        "filter[date_published][gte]": f"{year}-01-01T00:00:00Z",
        "filter[date_published][lte]": f"{year}-12-31T23:59:59Z",
        "page[size]": 1,  # ここ重要 → 総数 = 最終ページ番号
    }
    if provider:
        params["filter[provider]"] = provider

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    last_link = data.get("links", {}).get("last")
    if not last_link:
        # データが0件の場合
        return 0

    q = parse_qs(urlparse(last_link).query)
    return int(q['page'][0])

def count_total(path):
    # 既取得件数（既存ファイルの行数）
    try:
        with open(path, "r", encoding="utf-8") as f:
            fetched = sum(1 for _ in f)
    except FileNotFoundError:
        logging.warning(f"File {path} does not exist.")
        fetched = 0
    return fetched

In [ ]:
# # first day of publishing is : 2016-07-13T18:05:10.341000

# start_date = f"1990-01-01T00:00:00Z"
# end_date = f"2025-12-31T23:59:59Z"

# # 1ページあたりの件数（最大100）
# params = {
#     "page[size]": 5,
#     "filter[date_created][gte]": start_date,
#     "filter[date_created][lte]": end_date,
#     "sort": "date_created",
# }
# response = requests.get(url, params=params)
# data = response.json()
# data

{'data': [{'id': 'mvhkc_v1',
   'type': 'preprints',
   'attributes': {'date_created': '2016-07-13T18:05:10.341000',
    'date_modified': '2023-04-26T14:09:04.791207',
    'date_published': '2016-07-13T18:05:10.341000',
    'original_publication_date': None,
    'custom_publication_citation': None,
    'doi': '10.1177/0002716215587673',
    'title': 'How Troubling is our Inheritance - A review of genetics and race in the social sciences',
    'description': 'This article addresses the argument that there is variation between races in the biological basis for social behavior. The article uses Nicholas Wade’s popular book, A Troublesome Inheritance, as the point of departure for a discussion of attendant issues, including the extent to which human races can be definitively demarcated biologically, the extent to which genetics is related to contemporary definitions of race, and the role of natural selection as a possible mechanism for change in modern societies. My critical review of the 

In [ ]:
import time


for year in range(2016, 2025):
    path = f"{BASE_PATH}osf_preprints_{year}.jsonl"
    PAGE_SIZE = 100
    
    fetched = count_total(path)
    total = get_total_count(year)
    logging.info(f"Year {year}: total={total}, already have={fetched}")

    # 全取得済みならスキップ
    if fetched >= total:
        logging.info(f"Skipping year {year}: already complete.")
        continue

    # どの page から再開すべきか
    start_page = fetched // PAGE_SIZE + 1
    print(start_page)
    logging.info(f"Resuming from page {start_page} for year {year}")

    params = {
        "page[size]": PAGE_SIZE,
        "page": start_page,
        "filter[date_created][gte]": f"{year}-01-01T00:00:00Z",
        "filter[date_created][lte]": f"{year}-12-31T23:59:59Z",
        "sort": "date_created",
    }
    headers = {
        "Authorization": f"Bearer {API_TOKEN}"
    }

    url = BASE_URL
    results = []
    total_written = fetched  # 再開時点からカウント開始

    while url:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()

        results.extend(data["data"])

        if len(results) >= 1000:
            logging.info(f"Flushing {len(results)} results (year {year})")
            results = write_flush_file(results, path)

        # 次ページへ
        url = data["links"].get("next", None)
        params = None  # 2ページ目以降はクエリつけない
        time.sleep(1)  # API 過負荷防止のため少し待つ
    if results:
        logging.info(f"Flushing remaining {len(results)} results (year {year})")
        results = write_flush_file(results, path)

    # 取得数を更新
    total_written = sum(1 for _ in open(path, "r", encoding="utf-8"))
    logging.info(f"Finished year {year}: now have {total_written}/{total}")

115


In [ ]:
import json

def dedup_jsonl_by_id(input_path, output_path):
    seen = set()
    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            obj = json.loads(line)
            pid = obj["id"]  # OSF preprint ID
            if pid in seen:
                continue
            seen.add(pid)
            fout.write(line)

# 例

for i in range(2016, 2025):
    dedup_jsonl_by_id(f"{BASE_PATH}osf_preprints_{i}.jsonl", f"{BASE_PATH}osf_preprints_{i}.dedup.jsonl")
    path = f"{BASE_PATH}osf_preprints_{i}.dedup.jsonl"
    print("Completed deduplication for year", i, count_total(path))
    

1031
Completed deduplication for year 2016
5777
Completed deduplication for year 2017
12531
Completed deduplication for year 2018
23450
Completed deduplication for year 2019
24738
Completed deduplication for year 2020
24499
Completed deduplication for year 2021
24513
Completed deduplication for year 2022
24997
Completed deduplication for year 2023
15576
Completed deduplication for year 2024


In [ ]:
import json
import polars as pl

def date_formatter(date_str):
    if date_str is None:
        return None
    return date_str.split("T")[0]

if os.path.exists(BASE_PATH + 'osf_preprints.dedup.jsonl'):
    os.remove(BASE_PATH + 'osf_preprints.dedup.jsonl')

for year in range(2016, 2025):
    rows = []
    with open(BASE_PATH + f'osf_preprints_{year}.dedup.jsonl', 'r') as f:
        for line in f:
            obj = json.loads(line)
            rows.append({
                'id': obj['id'],
                'doi': obj['links']['preprint_doi'].replace("https://doi.org/", ""),
                # 'version': obj['attributes']['version'],
                # 'is_latest_version': obj['attributes']['is_latest_version'],
                'primary_category': obj['attributes']['subjects'][0][0]["text"] if obj['attributes']['subjects'] else None,
                'curated_doi': obj['attributes']['doi'],
                'published_date': date_formatter(obj['attributes']['date_published']),
                'last_updated_date': date_formatter(obj['attributes']['date_modified']),
                'server': obj['relationships']['provider']['data']['id'],
            })
    df = pl.DataFrame(rows)
    with open(BASE_PATH + f'osf_preprints.dedup.jsonl', 'a') as f:
        f.write(df.write_ndjson())

In [23]:
df = pl.scan_ndjson(BASE_PATH + 'osf_preprints.dedup.jsonl')
df.head().collect(engine="streaming")

id,doi,primary_category,curated_doi,published_date,last_updated_date,server
str,str,str,str,str,str,str
"""mvhkc_v1""","""10.31235/osf.io/mvhkc""","""Social and Behavioral Sciences""","""10.1177/0002716215587673""","""2016-07-13""","""2023-04-26""","""socarxiv"""
"""7y5rz_v1""","""10.31235/osf.io/7y5rz""","""Social and Behavioral Sciences""",null,"""2016-07-13""","""2018-07-02""","""socarxiv"""
"""4dymx_v1""","""10.31235/osf.io/4dymx""","""Social and Behavioral Sciences""","""10.1177/2329496515604170""","""2016-07-13""","""2023-04-26""","""socarxiv"""
"""4dws8_v1""","""10.31235/osf.io/4dws8""","""Social and Behavioral Sciences""",null,"""2016-07-13""","""2018-07-02""","""socarxiv"""
"""mrq9v_v1""","""10.31235/osf.io/mrq9v""","""Social and Behavioral Sciences""",null,"""2016-07-13""","""2018-07-02""","""socarxiv"""


In [31]:
print(df.filter(server="osf").unique("primary_category").collect(engine='streaming').get_column("primary_category").to_list())
print(df.filter(pl.col("server")!="osf").unique("primary_category").collect(engine='streaming').get_column("primary_category").to_list())


['Physical Sciences and Mathematics', None, 'Business', 'Education', 'Law', 'Social and Behavioral Sciences', 'Arts and Humanities', 'Engineering', 'Medicine and Health Sciences', 'Architecture', 'Life Sciences']
['Physical Sciences and Mathematics', 'Education', 'Life Sciences', 'Engineering', None, 'Rehabilitation and Therapy', 'Psychiatry', 'Chronic Venous Insufficiency', 'Compression', 'Pelvic Congestion Syndrome', 'Chronic Venous Disease', 'Surgery / Endovenous', 'Neuroscience', 'Business', 'Engineering Psychology', 'Sport and Exercise Science', 'Sport and Exercise Studies', 'Medieval Studies', 'Areas or Regions', 'Case Reports', 'Time Periods', 'Phlebitis', 'Law', 'Arts and Humanities', 'Medicine and Health Sciences', 'Aesthetic Phlebology', 'Theatre and Performance Studies', 'Social and Behavioral Sciences', 'Earth and Life Sciences', 'Meta-science', 'Architecture', 'General Subjects and Opinions', 'Sclerotherapy']


In [ ]:
['Physical Sciences and Mathematics', None, 'Business', 'Education', 'Law', 'Social and Behavioral Sciences', 'Arts and Humanities', 'Engineering', 'Medicine and Health Sciences', 'Architecture', 'Life Sciences']
['Physical Sciences and Mathematics', 'Education', 'Life Sciences', 'Engineering', None, 'Rehabilitation and Therapy', 'Psychiatry', 'Chronic Venous Insufficiency', 'Compression', 'Pelvic Congestion Syndrome', 'Chronic Venous Disease', 'Surgery / Endovenous', 'Neuroscience', 'Business', 'Engineering Psychology', 'Sport and Exercise Science', 'Sport and Exercise Studies', 'Medieval Studies', 'Areas or Regions', 'Case Reports', 'Time Periods', 'Phlebitis', 'Law', 'Arts and Humanities', 'Medicine and Health Sciences', 'Aesthetic Phlebology', 'Theatre and Performance Studies', 'Social and Behavioral Sciences', 'Earth and Life Sciences', 'Meta-science', 'Architecture', 'General Subjects and Opinions', 'Sclerotherapy']

In [28]:
print(df.count().collect(engine="streaming").select("id"))
print(df.with_columns(
    pl.col("id").is_duplicated().sum()
).select("id").head(1).collect(engine="streaming")    )

shape: (1, 1)
┌────────┐
│ id     │
│ ---    │
│ u32    │
╞════════╡
│ 157112 │
└────────┘
shape: (1, 1)
┌─────┐
│ id  │
│ --- │
│ u32 │
╞═════╡
│ 0   │
└─────┘


In [29]:
text = """
Year 2016: total=1030
Year 2017: total=5777
Year 2018: total=12492
Year 2019: total=23446
Year 2020: total=24762
Year 2021: total=24506
Year 2022: total=24508
Year 2023: total=24998
Year 2024: total=15463
"""
total_sum = 0
for line in text.strip().split("\n"):
    # sum the total
    year, total = line.split(": total=")
    print(f"{year.strip()}: {total.strip()}")
    total_sum += int(total.strip())
print("Total preprints from 2016 to 2024:", total_sum)

Year 2016: 1030
Year 2017: 5777
Year 2018: 12492
Year 2019: 23446
Year 2020: 24762
Year 2021: 24506
Year 2022: 24508
Year 2023: 24998
Year 2024: 15463
Total preprints from 2016 to 2024: 156982


In [ ]:
df.filter(pl.col('id').str.contains("v2"))

id,type,title,doi,version,is_latest_version,doi_preprint,is_published,is_preprint_orphan,date_published,provider
str,str,str,str,i64,bool,str,bool,bool,str,str


In [45]:
df.filter(id="mn7b6_v1")

id,type,title,doi,version,is_latest_version,doi_preprint,is_published,is_preprint_orphan,date_published,provider
str,str,str,str,i64,bool,str,bool,bool,str,str
"""mn7b6_v1""","""preprints""","""A Sociology of Archetypes""",null,1,false,"""https://doi.org/10.31234/osf.i…",true,false,"""2021-01-05T15:36:41.786965""","""psyarxiv"""


In [38]:
df.is_duplicated().sum()

200

In [5]:
df.unique("doi")

id,type,title,doi,version,is_latest_version,doi_preprint,is_published,is_preprint_orphan,date_published,provider
str,str,str,str,i64,bool,str,bool,bool,str,str
"""frx2w_v1""","""preprints""","""Language ERPs reflect learning…","""10.1016/j.cogpsych.2019.03.002""",1,true,"""https://doi.org/10.31234/osf.i…",true,false,"""2018-02-07T21:01:32.063444""","""psyarxiv"""
"""7a2c9_v1""","""preprints""","""It’s a Trust Thing: Assessing …","""10.1016/j.ocecoaman.2018.03.03…",1,true,"""https://doi.org/10.31230/osf.i…",true,false,"""2018-04-11T19:02:18.170045""","""marxiv"""
"""7caf4_v1""","""preprints""","""When Acute Adversity Improves …","""10.1037/rev0000144""",1,true,"""https://doi.org/10.31234/osf.i…",true,false,"""2018-07-24T15:43:34.803748""","""psyarxiv"""
"""ymnb6_v1""","""preprints""","""General Factors of Psychopatho…","""10.1177/216770217750150""",1,true,"""https://doi.org/10.31234/osf.i…",true,false,"""2018-03-27T21:20:20.047868""","""psyarxiv"""
"""abvus_v1""","""preprints""","""Public opposition and the neig…","""10.1016/j.landusepol.2018.08.0…",1,true,"""https://doi.org/10.31235/osf.i…",true,false,"""2018-10-05T03:54:43.627998""","""socarxiv"""
…,…,…,…,…,…,…,…,…,…,…
"""76tvu_v1""","""preprints""","""Mangrove pollen of Indonesia a…","""10.1016/j.margeo.2007.02.020""",1,true,"""https://doi.org/10.31223/osf.i…",null,null,"""2018-10-26T19:35:34.399426""","""eartharxiv"""
"""y9spq_v1""","""preprints""","""Contributorships are not weigh…","""10.1016/j.tibs.2016.03.001""",1,true,"""https://doi.org/10.31221/osf.i…",true,false,"""2018-01-30T20:18:37.736826""","""arabixiv"""
"""4j6rq_v1""","""preprints""","""""Thoroughly Reforming Them Tow…","""10.1080/02634937.2018.1507997""",1,true,"""https://doi.org/10.31235/osf.i…",true,false,"""2018-09-07T09:36:22.492369""","""socarxiv"""


In [8]:
df.filter(pl.col("id")=="frx2w_v1").select("doi_preprint").to_pandas()

,doi_preprint
0,https://doi.org/10.31234/osf.io/frx2w


In [23]:
import polars as pl
pl.read_ndjson(BASE_PATH + "osf_preprints_2020.jsonl")

ComputeError: expected dtype 'null' in JSON value, got dtype: Array

Encountered value: []